# Data leakage with the split by message
The idea is to check how the model's performance changes when the train, test and validation split is done "carelessly" for the format. In this case, the problem is that we split by messages at random, so part of the conversations are in validation and others in training. It's very likely that the model's performance will be better than in the tests done with the split by whole sessions. The hypothesis is that if the loss in that case was around 3.4 (or 3.2 depending on whether we account for the library bug or not), in this case it should be a lower number. How much lower is hard to calculate, so let's say around 2.8. Even if it already knows the conversations, going even lower means a risk of memorization, which we'll also check.


Remember that although the context is what lets it predict its response (what it learns), with --mask-prompt it isn't computed. So it only affects how accurate the response is in that context. Because of that, having part of the context leaked into train can help performance improve (but not for good reasons).

What leaks is my neighboring responses from the same afternoon, which it did learn to write. And we measure it in two ways: the same model on both tests, and both models on the same test.

> **Re-run on 7 Nov 2026 with ~1.9 epochs**, which was the pending item of decision 14: both models trained for 5 epochs with the same command and measured at step 2,500, the best step on validation for both (`by_message_5ep` and the final adapter `run1_5ep`). The first version used the 1,000-step runs (0.74 epochs); its numbers are kept in the conclusions for reference. The hypothesis above was written for that first version.

## Performance on test

The same measure as in Perplexity.ipynb (`nll_ejemplo`, without mlx's padding token). The final adapter on the session test should give loss 3.388 (ppl 29.6), as in Perplexity.ipynb: it's there to check that everything adds up.


In [1]:
import json, math, random
import numpy as np
import mlx.core as mx
from mlx_lm import load

MODELO = "mlx-community/Qwen2.5-0.5B-Instruct-4bit"
fuga, tokenizer = load(MODELO, adapter_path="../adapters/by_message_5ep_best")  # split by messages, step 2500 (its best on valid)
run1, _ = load(MODELO, adapter_path="../adapters/run1_5ep_best")     # split by sessions: the final adapter, step 2500

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

In [2]:
# Copied as is from Perplexity.ipynb
def nll_ejemplo(model, tokenizer, messages):
    full = tokenizer.apply_chat_template(messages)
    base_prompt = tokenizer.apply_chat_template(
        messages[:-1], add_generation_prompt=True)
    n_tokens = len(full) - len(base_prompt)
    logits = model(mx.array(full)[None])
    logits = logits[0, len(base_prompt)-1:len(full)-1]
    logits = logits.astype(mx.float32)
    logprobs = logits - mx.logsumexp(logits, axis=-1, keepdims=True)
    nll = 0
    for i in range(n_tokens):
        nll -= logprobs[i, full[len(base_prompt) + i]].item()
    mx.clear_cache()
    return (nll, n_tokens)

def medir(model, tokenizer, ruta):
    with open(ruta, "r") as f:
        ejemplos = [json.loads(line) for line in f if line.strip()]
    return [nll_ejemplo(model, tokenizer, e["messages"]) for e in ejemplos]

In [3]:
# Mean loss per token (over the whole folder) and its perplexity
def resumen(resultados):
    nll = sum(r[0] for r in resultados)
    n = sum(r[1] for r in resultados)
    return nll / n, math.exp(nll / n)

TEST_MENSAJES = "../data/dataset/by_message/test.jsonl"   # 506 examples, with leakage
TEST_SESIONES = "../data/dataset/session/test.jsonl"      # 1,133 examples, no model has seen it

# 1. The leakage: the SAME model (by_message) on both tests
r_fuga_mensajes = medir(fuga, tokenizer, TEST_MENSAJES)
r_fuga_sesiones = medir(fuga, tokenizer, TEST_SESIONES)

for nombre, r in [("by_message on its own test (messages)", r_fuga_mensajes),
                  ("by_message on session test", r_fuga_sesiones)]:
    loss, ppl = resumen(r)
    print(f"{nombre:34} loss {loss:.3f}  ppl {ppl:.1f}")

by_message on its own test (messages) loss 3.353  ppl 28.6
by_message on session test         loss 3.381  ppl 29.4


### Conclusion of the test
At ~1.9 epochs the leakage starts to show, but it's small: by_message gets 3.353 on its own test and 3.381 on the session test, a gap of 0.028 (with 0.74 epochs it was 3.405 against 3.423, 0.018). Nowhere near my prediction of 2.8. This notebook doesn't bootstrap, so I can't say whether 0.028 is beyond noise: as a reference, the session vs message split on validation differed by ~0.01-0.02 (decision 16).


## Better model or easier exam?

Both models on the session test, which neither has seen. If they come out similar, by_message isn't a better model: what went down was the exam.


In [4]:
# 2. Is it a better model or was the exam easier? Both models on the session test
r_run1_sesiones = medir(run1, tokenizer, TEST_SESIONES)  # should give loss 3.388 (ppl 29.6), as in Perplexity.ipynb

for nombre, r in [("by_message on session test", r_fuga_sesiones),
                  ("final adapter on session test", r_run1_sesiones)]:
    loss, ppl = resumen(r)
    print(f"{nombre:34} loss {loss:.3f}  ppl {ppl:.1f}")

by_message on session test         loss 3.381  ppl 29.4
run1 on session test               loss 3.388  ppl 29.6


### Conclusion of the test
The same: by_message 3.381 and the final adapter 3.388 on the session test (0.74 epochs: 3.423 and 3.421). They aren't really different models; the split by messages doesn't make a better model, it makes an easier exam, and only a little at this amount of training. My guess is still that most of the loss improvement comes from writing shorter messages, which the leaked neighbours don't help much with. Not tested yet.
